#**CHAPTER 2.TREES OF THOUGHT**
---

##REFERENCE

https://chatgpt.com/share/69985194-3b24-8012-8bd8-865aab82bc3f

##0.CONTEXT

**Board Briefing Memorandum — Tree Reasoning Under Governance (Chapter 2 Notebook)**

**Purpose of this paper**

This paper explains, in plain board-ready terms, what we are doing in the Chapter 2 notebook and why it matters. The notebook is not “an AI demo.” It is a controlled **reasoning pipeline**: a system that takes a bounded packet of inputs, evaluates a small set of strategic alternatives, records how it reasoned, enforces governance gates, and produces artifacts that a reviewer can inspect. The objective is simple: when a high-stakes decision is being considered, we want an analysis process that is **auditable, reproducible, and human-accountable** rather than a black box.

This notebook implements a specific reasoning architecture: **Tree Reasoning**. A tree is the correct shape when management and the board are not asking a single linear question (“Should we do X?”) but rather a comparative one (“Which path is best among several plausible paths?”). In corporate finance and investment banking contexts, we often decide among a small set of mutually exclusive strategic actions. The tree structure is how we represent that choice set explicitly and keep the evaluation disciplined: three paths in, one path out, with full traceability on what was considered and what was discarded.

**The strategic question and why the tree shape is appropriate**

The synthetic strategic question in this notebook is: **Buy competitor vs divest division vs do nothing**. This is a classic board-level decision pattern:

- **Acquire**: pursue scale, capabilities, and synergies, with integration risk and execution complexity.
- **Divest**: simplify, de-risk, or refocus, with carve-out complexity and stranded cost risk.
- **Do nothing**: preserve optionality and reduce disruption, with the risk of inertia and competitive drift.

The tree has exactly **three primary branches**—one per option. This is intentional. We cap the number of branches to prevent analysis drift. Boards do not need twenty scenarios; they need a **controlled comparative matrix** where the alternatives are distinct, the assumptions are explicit, and the next-step diligence questions are clear.

**What the notebook does, step by step: the reasoning pipeline**

The pipeline is built to behave like an engineered process rather than an ad-hoc spreadsheet plus narrative. In high-level terms, the pipeline does six things:

1) **Defines an input boundary**  
2) **Builds the reasoning tree (root + three branches)**  
3) **Evaluates each branch under deterministic scoring rules**  
4) **Adds constrained qualitative memos using the LLM**  
5) **Applies pruning rules (explicit, repeatable)**  
6) **Runs governance gates and produces a final report plus audit artifacts**

Each of these steps is recorded and persisted in artifacts.

**1) Input boundary: what the system is allowed to use (and what it is forbidden to use)**

The notebook uses a deterministic synthetic input packet. This is crucial: the point is not that the numbers are “real.” The point is that the system enforces correct behavior under governance constraints. We define:

- **facts_provided**: the bounded packet of inputs (baseline financials, business profile, execution risk factors, integration complexity).
- **assumption libraries**: pre-defined, branch-specific lists of assumptions the model is allowed to select from.
- **open item libraries**: pre-defined, branch-specific lists of diligence questions the model is allowed to select from.
- **no external data**: the system is explicitly prohibited from pulling market data, comparable multiples, or “facts not provided.”

This constraint is critical for safe board usage. Without it, a generative model will often fill gaps with plausible-sounding but unverified claims. The entire notebook is designed to prevent that and to surface uncertainty rather than hide it.

**2) Build the tree: a controlled topology**

The tree is minimal by design:

- Root node: **ROOT**
- Three child nodes: **Acquire**, **Divest**, **Do nothing**

There is no explosion of sub-branches. We cap nodes at four. This matters because uncontrolled branching is how analysis becomes unreviewable. In a board environment, “more scenarios” is not automatically better; “reviewable scenarios” is what we need.

Every node has a required schema, including:

- node_id, parent_id, label  
- facts_used (which keys from facts_provided were used)  
- assumptions (explicitly listed)  
- derived_metrics (a deterministic scorecard plus supporting fields)  
- risks (listed)  
- decision_status (KEEP or PRUNE)  
- prune_rationale (required if PRUNE)

This schema is not paperwork. It is the mechanism that forces discipline: every branch must state what it used, what it assumed, what it derived, and whether it survives.

**3) Deterministic evaluation: scorecards that are repeatable**

The core quantitative comparison is deterministic. That means that if you run the notebook again with the same inputs, you get the same scores. This eliminates the “randomness” risk that often makes AI outputs difficult to audit.

The scorecard includes:

- **value_score** (0–100): synthetic measure of potential value impact based on provided ranges and internal proxies.
- **risk_score** (0–100): synthetic measure of execution and integration risk derived from provided risk modifiers and complexity.
- **feasibility_score** (0–100): synthetic measure of operational executability given complexity.
- **overall_score**: weighted composite.

The weights are explicit in the config. The prune thresholds are explicit in the config. This makes the scoring approach inspectable. The board can challenge the weights and thresholds directly. That is a feature, not a bug: governance requires the ability to interrogate the model’s decision policy.

**4) Constrained qualitative analysis: what the LLM contributes**

The notebook uses Anthropic Claude (claude-haiku-4-5-20251001) to generate narrative content, but in a highly constrained way. The LLM is not allowed to “research.” It is not allowed to invent. It is used only to produce:

- a concise **thesis** per branch
- a selection of assumptions from the allowed list
- a list of risks (in text, but constrained)
- a selection of open items from the allowed list

Two constraints are particularly important:

- **Library membership enforcement**: the model can only choose assumptions and open items from pre-approved lists.
- **No-digits policy for LLM text** (in this notebook): this is a safety control to reduce the likelihood of invented numeric claims. It is not the only possible control, but it is a strong one for board-facing drafts.

This is how we use the LLM appropriately: it is a drafting assistant operating inside guardrails, not a free-form analyst that can fabricate.

**5) Pruning: explicit rules for rejecting branches**

Once branches are evaluated, the system applies pruning rules:

- prune if feasibility_score < threshold **OR**
- prune if risk_score > threshold

If a branch is pruned, the system must provide a prune rationale that cites the rule and the values. This is logged as audit evidence. A board should be able to see, clearly, “this option was discarded because it violated an explicit threshold,” not “because the AI felt like it.”

The pruned branches and the rationale are persisted in:

- reasoning_trace.json (node-level fields)
- prune_log (explicit list of pruned branches with rules + values)
- final_report.json (pruned_branches_audit)

**6) Governance gates: when the system escalates to HUMAN_REVIEW**

This is the most important part from a governance perspective. The system is not allowed to quietly proceed if controls fail. The notebook implements gates that can force escalation:

- **Gate A: Branch coverage check**  
  Ensures the tree has exactly three branches (Acquire, Divest, Do nothing). If not, the system flags a governance failure and escalates.

- **Gate B: Pruning justification check**  
  Ensures every PRUNE has a rationale that cites the explicit rule and the scores. If pruning is not justified, escalation occurs.

- **Gate C: No invented facts detector**  
  Ensures that LLM outputs contain no digits (per policy here) and that assumptions/open items are strict subsets of allowed libraries. If the model attempts to introduce content outside the allowed boundary, escalation occurs.

Additionally, schema validation is enforced for the reasoning trace and the final report. If schema validation fails, the system logs a risk and forces HUMAN_REVIEW. In other words: if the system cannot prove its output is structurally valid, it does not get to present itself as “ready.”

This is the governance philosophy made concrete: **capability implies risk; risk implies controls; controls imply enforceable gates.**

**What artifacts the board should care about**

The notebook produces a governance bundle every run. The relevant artifacts are:

- **run_manifest.json**  
  The run identity, environment fingerprint, config hash, determinism controls, and output paths. This is what lets an auditor reproduce the run and confirm the same settings were used.

- **prompts_log.jsonl (redacted + hashed)**  
  A record that prompts and responses were logged without leaking secrets. This is essential for traceability without compromising confidentiality.

- **reasoning_trace.json**  
  The full tree: every node, its scorecard, its assumptions, its decision status, and pruning rationale. This is the “audit trail” of the reasoning topology.

- **risk_log.json**  
  The formal governance record. If a gate fails or a policy concern is detected, it is recorded with severity, category, description, control, and status.

- **final_report.json**  
  The board-facing outcome: executive summary, scenario matrix, recommendation, confidence, open items, and verification status.

- **deliverables.zip**  
  A packaged bundle that can be handed to a reviewer.

This artifact structure matters because it turns an AI-assisted analysis into something closer to a controlled internal process. It becomes possible to answer: “What did we do, why did we do it, and what did we not do?”

**What results the pipeline produces and how to interpret them**

The pipeline produces two layers of results:

1) **Scenario matrix**: a comparative table of Acquire vs Divest vs Do nothing, including value/risk/feasibility and overall score, plus KEEP/PRUNE status.
2) **Recommendation object**: a recommended branch (or HUMAN_REVIEW), a decision label, and the rationale.

A key point: the decision is not presented as “truth.” The report’s verification_status is always **Not verified**. This is non-negotiable. The system is designed to support decision-making discipline, not to replace diligence.

The confidence field is also deliberately conservative. It is not “probabilistic certainty.” It is an internal classification (low/medium/high) with a rationale tied to score separation and governance posture. In production, the confidence logic can be tightened further and tied to independent validation.

**Why this is relevant to board decision-making**

Boards face two recurring problems with complex strategic decisions:

- **Process opacity**: Analysts and teams produce outputs, but the board cannot easily see the reasoning chain, the assumptions, or the decision logic.
- **Assumption drift**: Over time, assumptions become implicit. The narrative becomes persuasive, but the underlying uncertainty is forgotten.

This notebook addresses both problems.

First, it makes the process explicit. A tree is not a metaphor; it is a structured object with enforceable fields. Second, it isolates the most important governance requirement in finance: **separate facts from assumptions** and preserve open items. The system does not allow the model to smuggle in “facts.” It forces open questions to remain open.

In a board context, this is valuable because it supports the board’s role: not to compute the score, but to challenge the policy. The board can ask:

- Are these prune thresholds appropriate for our risk appetite?
- Are we weighting feasibility correctly relative to value?
- What diligence items are truly gating?
- What would flip the decision?

The system is a decision discipline amplifier.

**What this notebook is not (and why that matters)**

It is not an oracle. It does not provide market comps. It does not claim to know competitor pricing, valuation multiples, or regulatory outcomes. If it did, it would be unsafe. The notebook is intentionally designed to avoid that class of failure.

It is also not a substitute for investment committee judgment. The correct output is often HUMAN_REVIEW, and the system is designed to trigger that outcome when governance gates fail.

**How we would use this in practice**

In a real deployment, the synthetic inputs would be replaced by a bounded internal data packet approved for use. The same boundaries would apply:

- explicitly enumerated facts_provided
- approved assumption libraries (or explicitly signed-off assumptions)
- open item libraries aligned to diligence workstreams
- strict gating and artifact logging

The report would be used as an agenda-setting memo, not a final answer. It would help management and the board align on:

- which path is directionally favored
- what must be verified before committing resources
- what risks are gating
- what cannot be claimed yet

**Why the governance-first framing is the point**

The big value proposition is not that AI can “write faster.” The value is that AI can be embedded inside a process that is:

- repeatable (deterministic scoring)
- inspectable (trace + schema)
- safe (no invented facts controls)
- accountable (risk logs + escalation)
- portable (deliverables bundle)

That is what turns generative AI from a novelty into an institutional tool.

**Conclusion and next steps**

This notebook demonstrates a governance-first Tree Reasoning architecture that fits board-level comparative decisions. It shows how to:

- evaluate three strategic options under a controlled structure
- compute a repeatable scorecard
- draft constrained narratives without inventing facts
- prune options using explicit thresholds
- escalate to HUMAN_REVIEW when controls fail
- produce audit-ready artifacts every run

The next steps to move toward production-grade usage are straightforward:

- Replace synthetic packet with a governed internal input schema and data ingestion controls.
- Expand the no-invented-facts gate beyond “no digits” into claim-level grounding checks (still within governance boundaries).
- Add policy configuration management and approval workflows for weights, thresholds, and assumption libraries.
- Integrate human sign-off stages: analyst, risk, compliance, and final committee.
- Maintain artifact retention and reproducibility standards as a formal control requirement.

The board should view this as a prototype of a governed decision-support process: not “AI making decisions,” but “AI making our reasoning auditable.”

##1.LIBRARIES AND ENVIRONMENT

**CELL 1/10 — Install, deterministic setup, and controlled workspace**

This cell turns a blank Colab runtime into a predictable, governed workspace. First, it installs two pinned dependencies: the Anthropic SDK (to call Claude) and jsonschema (to enforce output structure). Pinning matters because governance depends on reproducibility. If library versions drift, subtle behavior changes can break outputs, validation, or logging. Next, the cell imports only the standard modules needed for a controlled pipeline: hashing, timestamps, filesystem operations, typing, and simple utilities. There is no hidden “magic” layer.

Then the cell establishes determinism controls. It sets a fixed random seed and a fixed PYTHONHASHSEED. The goal is that synthetic case generation and any hash-based operations behave consistently across runs. This is not perfect determinism in every aspect of Python, but it is an important baseline control for a notebook that aims to be auditable and repeatable.

After that, the cell creates a standard directory layout: an `artifacts/` folder for governance outputs and a `deliverables/` folder for the packaged bundle. This matters because the notebook is built around producing a structured set of files every run. Without deterministic paths, artifact integrity becomes fragile and review becomes harder.

Finally, it defines a single timestamp helper that always uses UTC in ISO format, and it generates a unique run identifier. Those two fields—run_id and run timestamp—become the backbone of the audit trail. They allow an independent reviewer to locate exactly which artifacts belong to which run, and to compare runs cleanly. The cell prints out the run identifiers and key environment fields so the operator can immediately see the run context. In short: Cell 1 establishes the “controlled lab bench” where everything else happens, with an emphasis on repeatability and traceability.

In [2]:
# CELL 1/10 — Install + imports + deterministic settings + directory setup
!pip -q install "anthropic==0.45.2" "jsonschema==4.23.0"

import os, json, re, uuid, hashlib, random, zipfile, platform
import datetime as _dt
from typing import Any, Dict, List, Optional, TypedDict, Literal, Tuple

from jsonschema import validate as _js_validate, Draft202012Validator
from jsonschema.exceptions import ValidationError
from google.colab import userdata

random.seed(7)
os.environ["PYTHONHASHSEED"] = "7"

BASE_DIR = os.getcwd()
ARTIFACTS_DIR = os.path.join(BASE_DIR, "artifacts")
DELIVERABLES_DIR = os.path.join(BASE_DIR, "deliverables")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(DELIVERABLES_DIR, exist_ok=True)

def utc_now_iso() -> str:
    return _dt.datetime.now(_dt.timezone.utc).isoformat()

RUN_ID = str(uuid.uuid4())
RUN_TS_UTC = utc_now_iso()

print("RUN_ID:", RUN_ID)
print("RUN_TS_UTC:", RUN_TS_UTC)
print("BASE_DIR:", BASE_DIR)
print("PYTHONHASHSEED:", os.environ.get("PYTHONHASHSEED"))

RUN_ID: bd625705-c304-4654-a746-c493eae18dc6
RUN_TS_UTC: 2026-02-20T11:44:43.089401+00:00
BASE_DIR: /content
PYTHONHASHSEED: 7


##2.CONFIGURATION AND SCHEMAS

###2.1.OVERVIEW

**CELL 2/10 — Governance configuration, schemas, and core safety helpers**

This cell defines the notebook’s “policy layer.” It creates a single CONFIG object that contains the model name, token and temperature settings, pruning thresholds, scoring weights, and hard caps on branching. In a governance-first system, these are not informal preferences—they are explicit parameters that determine what the system is allowed to do. By hashing the config, the notebook can prove later that a given report was produced under a specific policy version.

Next, the cell defines foundational helper functions used across the pipeline: secure hashing for text and JSON objects, and structured JSON writing utilities. Hashing is essential for audit logs: it lets us record that a prompt or config existed in a specific form without necessarily retaining sensitive content. The `write_json` function standardizes how artifacts are written: consistent formatting, consistent sorting, consistent encoding—small details that matter for reproducibility and downstream diffing.

The cell also defines a redaction function. Even though this is a synthetic notebook, the governance posture is “never leak secrets.” Redaction patterns remove anything resembling an API key, email, or phone-like pattern. This ensures the prompts log can be stored and reviewed without exposing sensitive strings. For a board-facing workflow, this is a non-negotiable control: an audit trail must not become a data-leak path.

Most importantly, this cell defines JSON Schemas for the key outputs: tree nodes, the reasoning trace, and the final report. Schemas are the enforcement mechanism that prevents the system from emitting loosely structured narratives that are hard to review. If schema validation fails, later cells will log a risk and force escalation. This is how we move from “AI output” to “controlled deliverable.” Cell 2 therefore establishes the formal contract that all later cells must satisfy: structure, constraints, and safety helpers that enable governance.

###2.2.CODE AND IMPLEMENTATION

In [6]:
# CELL 2/10 — Config + schemas + helpers (hashing, redaction, JSON writing, validation)

CONFIG: Dict[str, Any] = {
    "project": "Reasoning AI in Finance — Structured Inference Under Governance",
    "notebook": "Chapter 2 — Tree Reasoning (Multi-scenario investment banking decision)",
    "model": "claude-haiku-4-5-20251001",
    "max_llm_tokens": 700,
    "temperature": 0.0,
    "prune_rules": {
        "min_feasibility_score": 55,
        "max_risk_score": 70
    },
    "scoring_weights": {
        "value": 0.45,
        "risk": 0.35,
        "feasibility": 0.20
    },
    "hard_caps": {
        "max_nodes": 4,  # root + 3 branches only
        "max_branches": 3
    },
    "gates": {
        "branch_coverage_required": 3,
        "require_prune_justification": True,
        "no_digits_in_llm_text": True
    }
}

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def sha256_json(obj: Any) -> str:
    return sha256_text(json.dumps(obj, sort_keys=True, ensure_ascii=False))

_REDACT_PATTERNS = [
    (re.compile(r"sk-[A-Za-z0-9_\-]+"), "[REDACTED_API_KEY]"),
    (re.compile(r"(?i)anthropic_api_key\s*[:=]\s*['\"][^'\"]+['\"]"), "ANTHROPIC_API_KEY='[REDACTED]'"),
    (re.compile(r"(?i)\b\d{3}-\d{2}-\d{4}\b"), "[REDACTED_SSN]"),
    (re.compile(r"(?i)\b(?:\+?\d{1,3})?[-.\s]?\(?\d{2,3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b"), "[REDACTED_PHONE]"),
    (re.compile(r"(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b"), "[REDACTED_EMAIL]"),
]

def redact_text(s: str) -> str:
    out = s
    for pat, repl in _REDACT_PATTERNS:
        out = pat.sub(repl, out)
    return out

def write_json(path: str, obj: Any) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, sort_keys=True)

def append_jsonl(path: str, obj: Any) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False, sort_keys=True) + "\n")

def schema_check(obj: Any, schema: Dict[str, Any]) -> Tuple[bool, Optional[str]]:
    try:
        Draft202012Validator.check_schema(schema)
        _js_validate(instance=obj, schema=schema)
        return True, None
    except ValidationError as e:
        return False, f"{e.message}"
    except Exception as e:
        return False, str(e)

def mk_risk(risk_id: str, severity: str, category: str, description: str, control: str, status: str) -> Dict[str, Any]:
    return {
        "risk_id": risk_id,
        "timestamp_utc": utc_now_iso(),
        "severity": severity,
        "category": category,
        "description": description,
        "control": control,
        "status": status
    }

TREE_NODE_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "required": ["node_id", "parent_id", "label", "facts_used", "assumptions", "derived_metrics", "risks", "decision_status"],
    "properties": {
        "node_id": {"type": "string"},
        "parent_id": {"type": ["string", "null"]},
        "label": {"type": "string", "enum": ["ROOT", "Acquire", "Divest", "Do nothing"]},
        "facts_used": {"type": "array", "items": {"type": "string"}},
        "assumptions": {"type": "array", "items": {"type": "string"}},
        "derived_metrics": {
            "type": "object",
            "required": ["scorecard"],
            "properties": {
                "scorecard": {
                    "type": "object",
                    "required": ["value_score", "risk_score", "feasibility_score", "overall_score"],
                    "properties": {
                        "value_score": {"type": "integer", "minimum": 0, "maximum": 100},
                        "risk_score": {"type": "integer", "minimum": 0, "maximum": 100},
                        "feasibility_score": {"type": "integer", "minimum": 0, "maximum": 100},
                        "overall_score": {"type": "number"}
                    },
                    "additionalProperties": False
                },
                "thesis": {"type": "string"},
                "open_items": {"type": "array", "items": {"type": "string"}}
            },
            "additionalProperties": True
        },
        "risks": {"type": "array", "items": {"type": "string"}},
        "decision_status": {"type": "string", "enum": ["KEEP", "PRUNE"]},
        "prune_rationale": {"type": ["string", "null"]}
    },
    "additionalProperties": False
}

REASONING_TRACE_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "required": ["run_id", "timestamp_utc", "shape", "root_node_id", "nodes", "prune_log", "termination"],
    "properties": {
        "run_id": {"type": "string"},
        "timestamp_utc": {"type": "string"},
        "shape": {"type": "string", "enum": ["TREE"]},
        "root_node_id": {"type": "string"},
        "nodes": {"type": "array", "items": TREE_NODE_SCHEMA},
        "prune_log": {"type": "array", "items": {"type": "object"}},
        "termination": {"type": "object"}
    },
    "additionalProperties": False
}

FINAL_REPORT_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "required": [
        "run_id", "timestamp_utc", "executive_summary", "facts_provided", "assumptions_introduced",
        "analysis", "recommendation", "confidence", "open_items", "questions_to_verify", "verification_status",
        "scenario_matrix", "pruned_branches_audit"
    ],
    "properties": {
        "run_id": {"type": "string"},
        "timestamp_utc": {"type": "string"},
        "executive_summary": {"type": "string"},
        "facts_provided": {"type": "object"},
        "assumptions_introduced": {"type": "array", "items": {"type": "string"}},
        "analysis": {"type": "object"},
        "recommendation": {"type": "object"},
        "confidence": {"type": "object"},
        "open_items": {"type": "array", "items": {"type": "string"}},
        "questions_to_verify": {"type": "array", "items": {"type": "string"}},
        "verification_status": {"type": "string", "enum": ["Not verified"]},
        "scenario_matrix": {"type": "array", "items": {"type": "object"}},
        "pruned_branches_audit": {"type": "array", "items": {"type": "object"}}
    },
    "additionalProperties": False
}

print("CONFIG_HASH:", sha256_json(CONFIG))
print("Schemas ready.")

CONFIG_HASH: 3e9ee3936e476d6fc609ab9a1e4bc161b19407971beca871e1503e1037c4763f
Schemas ready.


##3.SYNTHETIC FINANCE CASE GENERATOR

###3.1.OVERVIEW

**CELL 3/10 — Deterministic synthetic case generation and input boundary definition**

This cell constructs the synthetic “board packet” that the reasoning system is allowed to use. The key concept is that the notebook does not start by letting the model roam freely; it starts by defining a bounded input object. The case includes baseline financials, business profile, execution risk factors, and an integration complexity score. These are treated as `facts_provided`, meaning they are the only permissible factual grounding for the analysis.

The cell also defines two critical governance devices: an “assumption library” and an “open item library,” both organized by branch (Acquire, Divest, Do nothing). This is intentional. We want the model to behave like an analyst who can propose assumptions and diligence questions, but only within a pre-approved menu. That prevents the model from introducing claims that look plausible but have not been vetted or agreed.

In addition, the cell defines “scoring inputs” such as synergy ranges and other synthetic anchors. These are explicitly labeled as assumptions and synthetic values—not market facts. Their purpose is to enable deterministic scoring later, not to pretend we have real market information. This is an important teaching point: in a governance-first pipeline, we can separate (a) what is real evidence, (b) what is policy, and (c) what is synthetic scaffolding used to demonstrate architecture.

Finally, the cell defines an explicit `input_boundary` object stating the rules: no external data, no market facts, and use only provided numbers. This becomes an enforceable contract used by later gates. Cell 3 therefore creates the entire “world” in which the system is allowed to reason. Everything the model produces must either come directly from these facts or be labeled as assumptions/open items drawn from the approved lists. That separation is the central safety property of the notebook.

###3.2.CODE AND IMPLEMENTATION

In [7]:
# CELL 3/10 — Synthetic finance case generator (deterministic) + input boundary definition

class SyntheticCase(TypedDict):
    case_id: str
    timestamp_utc: str
    strategic_question: str
    facts_provided: Dict[str, Any]
    assumption_library: Dict[str, List[str]]
    open_item_library: Dict[str, List[str]]
    scoring_inputs: Dict[str, Any]
    input_boundary: Dict[str, Any]

def generate_case(seed: int = 7) -> SyntheticCase:
    random.seed(seed)
    case_id = f"CASE-{seed}-TREE"

    facts_provided = {
        "baseline_financials": {
            "revenue_m": 420,
            "ebitda_m": 63,
            "ebitda_margin": 0.15,
            "net_debt_m": 160
        },
        "business_profile": {
            "sector": "industrial services",
            "end_markets": ["manufacturing", "logistics", "utilities"],
            "cyclicality": "medium"
        },
        "execution_risk_factors": {
            "regulatory_complexity": "medium",
            "customer_concentration": "moderate",
            "integration_history": "limited"
        },
        "integration_complexity_score": 62  # 0–100 (higher = harder)
    }

    assumption_library = {
        "Acquire": [
            "Synergies realized at low end of provided range",
            "Synergies realized at midpoint of provided range",
            "Integration timeline extends by one quarter",
            "One-time integration costs at midpoint of provided range"
        ],
        "Divest": [
            "Divested division sold at conservative multiple range (synthetic)",
            "Dis-synergies limited due to stand-alone readiness",
            "Management distraction limited via dedicated carve-out team"
        ],
        "Do nothing": [
            "Organic growth continues at baseline trend",
            "No major capital structure change",
            "Cost program delivers modest margin uplift"
        ]
    }

    open_item_library = {
        "Acquire": [
            "Confirm overlap-driven cost synergy feasibility by function",
            "Validate customer retention sensitivity during integration",
            "Clarify required IT systems integration scope"
        ],
        "Divest": [
            "Clarify stand-alone costs and TSA duration",
            "Validate buyer universe appetite and timing window",
            "Confirm stranded cost mitigation plan"
        ],
        "Do nothing": [
            "Validate pipeline durability and pricing power",
            "Confirm capex requirements under baseline plan",
            "Assess competitive response risk over next 12 months"
        ]
    }

    scoring_inputs = {
        "synergy_range_m": {"low": 6, "high": 18},                 # assumptions, not market facts
        "integration_complexity_score": facts_provided["integration_complexity_score"],
        "execution_risk_modifiers": {
            "regulatory_complexity": facts_provided["execution_risk_factors"]["regulatory_complexity"],
            "customer_concentration": facts_provided["execution_risk_factors"]["customer_concentration"],
            "integration_history": facts_provided["execution_risk_factors"]["integration_history"]
        },
        "divestiture_value_range_m": {"low": 45, "high": 85},     # synthetic anchor
        "status_quo_value_uplift_range_m": {"low": 8, "high": 20}  # synthetic anchor
    }

    input_boundary = {
        "allowed_facts_keys": list(facts_provided.keys()),
        "allowed_branch_labels": ["Acquire", "Divest", "Do nothing"],
        "no_external_data": True,
        "no_market_facts": True,
        "must_use_only_provided_numbers": True
    }

    return {
        "case_id": case_id,
        "timestamp_utc": utc_now_iso(),
        "strategic_question": "Buy competitor vs divest division vs do nothing",
        "facts_provided": facts_provided,
        "assumption_library": assumption_library,
        "open_item_library": open_item_library,
        "scoring_inputs": scoring_inputs,
        "input_boundary": input_boundary
    }

CASE: SyntheticCase = generate_case(seed=7)
print("CASE_ID:", CASE["case_id"])
print("Strategic question:", CASE["strategic_question"])
print("Facts keys:", list(CASE["facts_provided"].keys()))

CASE_ID: CASE-7-TREE
Strategic question: Buy competitor vs divest division vs do nothing
Facts keys: ['baseline_financials', 'business_profile', 'execution_risk_factors', 'integration_complexity_score']


##4.LLM CLIENT WRAPPPER

###4.1.OVERVIEW

**CELL 4/10 — LLM wrapper with redacted prompt logging and strict schema expectations**

This cell turns the LLM into a controlled component rather than a free-form oracle. It begins by retrieving the API key from Colab Secrets and failing hard if the key is missing. That is both operational hygiene and governance: we should never accidentally proceed in a partially configured state.

The wrapper is built around two principles: (1) every LLM interaction is logged in a redacted form, and (2) the model is asked to produce strict JSON that can be validated. The cell defines an output schema for branch memos: label, thesis, selected assumptions, risks, and selected open items. This schema is intentionally narrow. We do not ask the model for valuations, market research, or numerical conclusions. We ask it for constrained narrative elements and controlled selections.

The prompt itself includes explicit rules: no invented facts, no external references, and assumptions/open items must be chosen only from the provided lists. The wrapper logs the prompt and response in `prompts_log.jsonl` using two controls: redaction (removing sensitive patterns) and hashing (so we can verify integrity even if we truncate stored text). This supports auditability without leaking secrets.

A particularly strong control in this notebook is the “no digits” rule for LLM text. The idea is to reduce the chance that the model introduces new numbers as if they were facts. It is not the only approach in production, but it is a simple and robust safeguard in a teaching notebook.

In short, Cell 4 establishes the LLM as a bounded drafting engine: it can write a thesis and list risks, but only within guardrails. It also establishes a traceable audit trail of LLM usage. This is the key shift from casual prompting to governed usage: you can always answer “what did we ask the model, what did it answer, and how do we know it complied with constraints?”

###4.2.CODE AND IMPLEMENTATION

In [8]:
# CELL 4/10 — LLM client wrapper (Anthropic) + prompt logging (redacted + hashes)

from anthropic import Anthropic

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY or not isinstance(ANTHROPIC_API_KEY, str) or len(ANTHROPIC_API_KEY) < 10:
    raise RuntimeError("Missing/invalid ANTHROPIC_API_KEY in Colab Secrets.")

_client = Anthropic(api_key=ANTHROPIC_API_KEY)

PROMPTS_LOG_PATH = os.path.join(ARTIFACTS_DIR, "prompts_log.jsonl")

LLM_BRANCH_OUTPUT_SCHEMA: Dict[str, Any] = {
    "type": "object",
    "required": ["label", "thesis", "assumptions_selected", "risks", "open_items_selected"],
    "properties": {
        "label": {"type": "string", "enum": ["Acquire", "Divest", "Do nothing"]},
        "thesis": {"type": "string"},
        "assumptions_selected": {"type": "array", "items": {"type": "string"}},
        "risks": {"type": "array", "items": {"type": "string"}},
        "open_items_selected": {"type": "array", "items": {"type": "string"}}
    },
    "additionalProperties": False
}

def _contains_digits(s: str) -> bool:
    return bool(re.search(r"\d", s or ""))

def llm_branch_memo(case: SyntheticCase, label: str) -> Dict[str, Any]:
    allowed_assumptions = case["assumption_library"][label]
    allowed_open = case["open_item_library"][label]

    system = (
        "You are a finance analyst producing a board-ready scenario memo. "
        "You must NOT introduce any facts or numbers not provided. "
        "You must not use any digits in your thesis. "
        "Select assumptions and open items ONLY from the provided lists."
    )

    user = {
        "task": "Produce a concise scenario memo for a single branch in a controlled decision tree.",
        "strategic_question": case["strategic_question"],
        "branch_label": label,
        "facts_provided": case["facts_provided"],
        "allowed_assumptions": allowed_assumptions,
        "allowed_open_items": allowed_open,
        "output_instructions": {
            "format": "strict_json",
            "schema": "LLM_BRANCH_OUTPUT_SCHEMA",
            "rules": [
                "No digits anywhere in 'thesis' or risk texts",
                "assumptions_selected must be subset of allowed_assumptions",
                "open_items_selected must be subset of allowed_open_items",
                "No external references, no market data, no invented facts"
            ]
        }
    }

    prompt_text = json.dumps(user, ensure_ascii=False, sort_keys=True)
    redacted_prompt = redact_text(prompt_text)

    append_jsonl(PROMPTS_LOG_PATH, {
        "run_id": RUN_ID,
        "timestamp_utc": utc_now_iso(),
        "model": CONFIG["model"],
        "purpose": "branch_memo",
        "branch_label": label,
        "prompt_sha256": sha256_text(redacted_prompt),
        "prompt_redacted": redacted_prompt[:2000]
    })

    msg = _client.messages.create(
        model=CONFIG["model"],
        max_tokens=CONFIG["max_llm_tokens"],
        temperature=CONFIG["temperature"],
        system=system,
        messages=[{"role": "user", "content": redacted_prompt}],
    )

    raw = msg.content[0].text if msg and msg.content else ""
    raw_redacted = redact_text(raw)

    append_jsonl(PROMPTS_LOG_PATH, {
        "run_id": RUN_ID,
        "timestamp_utc": utc_now_iso(),
        "model": CONFIG["model"],
        "purpose": "branch_memo_response",
        "branch_label": label,
        "response_sha256": sha256_text(raw_redacted),
        "response_redacted": raw_redacted[:2000]
    })

    try:
        parsed = json.loads(raw_redacted)
    except Exception:
        parsed = {"_parse_error": True, "_raw": raw_redacted}

    return parsed

print("LLM wrapper ready.")

LLM wrapper ready.


##5.REASONING ENGINE CORE

###5.1.OVERVIEW

**CELL 5/10 — Tree builder, deterministic evaluation, and pruning engine (the core reasoning mechanism)**

This cell is the heart of Chapter 2: it implements Tree Reasoning as actual code. The most important design choice is that the tree topology is fixed and small: one root node and exactly three branch nodes. This prevents “branch explosion,” which is a common failure mode where analysis becomes unreviewable. The `build_tree` function produces a predictable, capped structure with stable node IDs. That stability matters for auditability: node N1 always refers to the same branch in a run.

Next, the cell defines deterministic scoring rules. These convert the synthetic case inputs into a comparable scorecard for each branch. The scoring is not meant to be “truth.” It is meant to be transparent and repeatable. It uses simple proxies: synergy ranges, integration complexity, leverage proxy, and qualitative modifiers mapped into numeric adjustments. This is deliberately inspectable. A board or reviewer can challenge the rule logic and weights. That is exactly the point: the scoring policy is explicit and can be governed.

The `evaluate_node` function integrates the two worlds: deterministic scoring plus constrained LLM memo content. It attaches the thesis, selected assumptions, and open items to the node, but does not let the LLM override the scorecard. This preserves the idea that narrative is supportive, while the comparative rubric remains stable and auditable.

Finally, `prune_tree` applies explicit thresholds. Pruning is not subjective: prune if feasibility is below the minimum or if risk exceeds the maximum. Crucially, pruning requires a rationale that cites the rule and the observed values. This creates a machine-checkable audit trail. The prune log is not “because the model said so.” It is “because the rule was triggered with specific scores.” Cell 5 therefore operationalizes Tree Reasoning: build, evaluate, prune—under explicit policy.

###5.2.CODE AND IMPLEMENTATION

In [9]:
# CELL 5/10 — Reasoning engine core: TREE BUILDER (build_tree / evaluate_node / prune_tree)

class TreeNode(TypedDict, total=False):
    node_id: str
    parent_id: Optional[str]
    label: Literal["ROOT", "Acquire", "Divest", "Do nothing"]
    facts_used: List[str]
    assumptions: List[str]
    derived_metrics: Dict[str, Any]
    risks: List[str]
    decision_status: Literal["KEEP", "PRUNE"]
    prune_rationale: Optional[str]

class TreeBuildResult(TypedDict):
    root_node_id: str
    nodes: List[TreeNode]

def _clamp_int(x: int, lo: int = 0, hi: int = 100) -> int:
    return max(lo, min(hi, int(x)))

def _map_modifier(val: str, mapping: Dict[str, int], default: int) -> int:
    return mapping.get((val or "").lower().strip(), default)

def _deterministic_scores(case: SyntheticCase, label: str) -> Dict[str, Any]:
    si = case["scoring_inputs"]
    base = case["facts_provided"]["baseline_financials"]
    complexity = int(si["integration_complexity_score"])

    rev = int(base["revenue_m"])
    ebitda = int(base["ebitda_m"])
    net_debt = int(base["net_debt_m"])

    ebitda_margin = float(base["ebitda_margin"])
    leverage_proxy = (net_debt / ebitda) if ebitda > 0 else 99.0

    # Deterministic scoring rules (no market facts)
    if label == "Acquire":
        # Value: benefits from synergy range; penalize complexity and leverage
        sy_low = int(si["synergy_range_m"]["low"])
        sy_high = int(si["synergy_range_m"]["high"])
        sy_mid = (sy_low + sy_high) / 2.0
        value = 55 + int((sy_mid / max(1, ebitda)) * 40)  # synergy relative to EBITDA
        risk = 35 + int(complexity * 0.6) + int(max(0.0, leverage_proxy - 2.0) * 8)
        feasibility = 80 - int(complexity * 0.5)
    elif label == "Divest":
        dv_low = int(si["divestiture_value_range_m"]["low"])
        dv_high = int(si["divestiture_value_range_m"]["high"])
        dv_mid = (dv_low + dv_high) / 2.0
        # Value: benefit from deleveraging / focus; penalize if EBITDA margin is low (indicative)
        value = 60 + int((dv_mid / max(1, net_debt)) * 25) + int(ebitda_margin * 20)
        risk = 40 + int(complexity * 0.25)
        feasibility = 70 - int(complexity * 0.15)
    else:  # Do nothing
        su_low = int(si["status_quo_value_uplift_range_m"]["low"])
        su_high = int(si["status_quo_value_uplift_range_m"]["high"])
        su_mid = (su_low + su_high) / 2.0
        value = 50 + int((su_mid / max(1, ebitda)) * 35) + int(ebitda_margin * 10)
        risk = 45 + int(complexity * 0.15) + int(max(0.0, leverage_proxy - 2.0) * 5)
        feasibility = 85 - int(complexity * 0.10)

    # Execution risk modifiers (qualitative -> numeric)
    mods = si["execution_risk_modifiers"]
    reg = _map_modifier(mods.get("regulatory_complexity", ""), {"low": -4, "medium": 0, "high": 6}, 0)
    conc = _map_modifier(mods.get("customer_concentration", ""), {"low": -3, "moderate": 2, "high": 7}, 2)
    hist = _map_modifier(mods.get("integration_history", ""), {"strong": -4, "good": -2, "limited": 4, "none": 6}, 4)
    risk = risk + reg + conc + (hist if label == "Acquire" else int(hist * 0.3))

    value_score = _clamp_int(value)
    risk_score = _clamp_int(risk)
    feasibility_score = _clamp_int(feasibility)

    w = CONFIG["scoring_weights"]
    overall = round((value_score * w["value"]) + ((100 - risk_score) * w["risk"]) + (feasibility_score * w["feasibility"]), 2)

    return {
        "scorecard": {
            "value_score": value_score,
            "risk_score": risk_score,
            "feasibility_score": feasibility_score,
            "overall_score": overall
        }
    }

def build_tree(case: SyntheticCase) -> TreeBuildResult:
    root_id = "N0"
    root: TreeNode = {
        "node_id": root_id,
        "parent_id": None,
        "label": "ROOT",
        "facts_used": list(case["facts_provided"].keys()),
        "assumptions": [],
        "derived_metrics": {"scorecard": {"value_score": 0, "risk_score": 0, "feasibility_score": 0, "overall_score": 0.0}},
        "risks": [],
        "decision_status": "KEEP",
        "prune_rationale": None
    }

    branches = ["Acquire", "Divest", "Do nothing"]
    nodes: List[TreeNode] = [root]

    for i, b in enumerate(branches, start=1):
        nid = f"N{i}"
        node: TreeNode = {
            "node_id": nid,
            "parent_id": root_id,
            "label": b,  # type: ignore
            "facts_used": list(case["facts_provided"].keys()),
            "assumptions": [],
            "derived_metrics": {},
            "risks": [],
            "decision_status": "KEEP",
            "prune_rationale": None
        }
        nodes.append(node)

    # Hard cap
    nodes = nodes[:CONFIG["hard_caps"]["max_nodes"]]
    return {"root_node_id": root_id, "nodes": nodes}

def evaluate_node(node: TreeNode, case: SyntheticCase, llm_payload: Optional[Dict[str, Any]] = None) -> TreeNode:
    label = node["label"]
    if label == "ROOT":
        return node

    # Deterministic metrics
    node["derived_metrics"] = _deterministic_scores(case, str(label))

    # LLM qualitative memo (constrained)
    if llm_payload:
        node["derived_metrics"]["thesis"] = str(llm_payload.get("thesis", "")).strip()
        node["assumptions"] = list(llm_payload.get("assumptions_selected", []))
        node["risks"] = list(llm_payload.get("risks", []))
        node["derived_metrics"]["open_items"] = list(llm_payload.get("open_items_selected", []))
    else:
        node["derived_metrics"]["thesis"] = ""
        node["derived_metrics"]["open_items"] = []
        node["assumptions"] = []
        node["risks"] = []

    return node

def prune_tree(nodes: List[TreeNode]) -> Tuple[List[TreeNode], List[Dict[str, Any]]]:
    prune_log: List[Dict[str, Any]] = []
    min_feas = int(CONFIG["prune_rules"]["min_feasibility_score"])
    max_risk = int(CONFIG["prune_rules"]["max_risk_score"])

    for node in nodes:
        if node["label"] == "ROOT":
            continue
        sc = (node.get("derived_metrics") or {}).get("scorecard") or {}
        feas = int(sc.get("feasibility_score", 0))
        risk = int(sc.get("risk_score", 0))

        if feas < min_feas or risk > max_risk:
            node["decision_status"] = "PRUNE"
            rule_hits = []
            if feas < min_feas:
                rule_hits.append(f"feasibility_score({feas}) < min_feasibility_score({min_feas})")
            if risk > max_risk:
                rule_hits.append(f"risk_score({risk}) > max_risk_score({max_risk})")
            rationale = "PRUNE_RULE: " + " OR ".join(rule_hits)
            node["prune_rationale"] = rationale

            prune_log.append({
                "node_id": node["node_id"],
                "label": node["label"],
                "timestamp_utc": utc_now_iso(),
                "rule": rationale,
                "values": {"feasibility_score": feas, "risk_score": risk}
            })
        else:
            node["decision_status"] = "KEEP"
            node["prune_rationale"] = None

    return nodes, prune_log

print("Tree engine ready.")

Tree engine ready.


##6.GATES AND RISK DETECTION

###6.1.OVERVIEW

**CELL 6/10 — Governance gates, risk detection, and escalation to HUMAN_REVIEW**

This cell implements the notebook’s safety enforcement. It defines gates that check whether the reasoning process stayed within policy. The gates matter because “good intentions” are not governance. Governance requires explicit checks that can fail and produce an escalation outcome.

Gate A verifies branch coverage: the tree must contain exactly three branches with the required labels. If the structure is incomplete or malformed, the system cannot claim it evaluated all options. In a board context, missing a branch is a serious control failure because the recommendation would be biased by omission.

Gate B verifies pruning justification. If a branch is marked PRUNE, the system must provide the prune rationale and include the scores used. This gate prevents silent pruning and ensures every discarded option has audit evidence. In regulated environments, the ability to explain why an option was excluded is essential.

Gate C is the “no invented facts” gate. In this notebook, it enforces that the LLM text contains no digits and that selected assumptions/open items belong to the approved libraries. This addresses the most common generative AI risk: hallucinated or ungrounded claims. By enforcing library membership, the system prevents the model from “inventing” diligence items that were never agreed.

The cell also defines the recommendation logic: it selects the best KEEP branch by overall score, but if all branches are pruned—or if any high-severity HUMAN_REVIEW risks exist—it forces escalation. This is the most important institutional behavior: a governance-first system must be able to say “I cannot safely decide; a human must review.” Cell 6 therefore translates governance from a concept into executable policy: gates, risks, and escalation rules.

###6.2.CODE AND IMPLEMENTATION

In [10]:
# CELL 6/10 — Gate(s) + risk detection + escalation logic

def gate_branch_coverage(nodes: List[TreeNode]) -> Tuple[bool, str]:
    labels = [n["label"] for n in nodes if n["label"] != "ROOT"]
    ok = (len(labels) == CONFIG["gates"]["branch_coverage_required"] and
          sorted(labels) == sorted(["Acquire", "Divest", "Do nothing"]))
    return ok, "PASS" if ok else "FAIL: branch coverage missing or incorrect"

def gate_pruning_justification(nodes: List[TreeNode]) -> Tuple[bool, str]:
    if not CONFIG["gates"]["require_prune_justification"]:
        return True, "PASS"
    for n in nodes:
        if n["label"] == "ROOT":
            continue
        if n["decision_status"] == "PRUNE":
            rat = (n.get("prune_rationale") or "").strip()
            sc = (n.get("derived_metrics") or {}).get("scorecard") or {}
            feas = sc.get("feasibility_score", None)
            risk = sc.get("risk_score", None)
            if (not rat.startswith("PRUNE_RULE:")) or (feas is None) or (risk is None):
                return False, f"FAIL: missing prune rationale or values for node {n['node_id']}"
    return True, "PASS"

def gate_no_invented_facts(nodes: List[TreeNode], case: SyntheticCase) -> Tuple[bool, str]:
    # Strict control: LLM outputs must (a) be valid schema, (b) use only allowed assumption/open lists, (c) contain no digits (optional)
    for n in nodes:
        if n["label"] == "ROOT":
            continue
        label = str(n["label"])
        thesis = str((n.get("derived_metrics") or {}).get("thesis", "") or "")
        risks = n.get("risks") or []
        open_items = (n.get("derived_metrics") or {}).get("open_items") or []
        assumptions = n.get("assumptions") or []

        # No digits check (prevents numeric invention)
        if CONFIG["gates"]["no_digits_in_llm_text"]:
            if _contains_digits(thesis) or any(_contains_digits(str(r)) for r in risks) or any(_contains_digits(str(x)) for x in open_items):
                return False, f"FAIL: digits detected in LLM text for node {n['node_id']}"

        allowed_assumptions = set(case["assumption_library"][label])
        allowed_open = set(case["open_item_library"][label])

        if any(a not in allowed_assumptions for a in assumptions):
            return False, f"FAIL: assumption not in allowed library for node {n['node_id']}"
        if any(o not in allowed_open for o in open_items):
            return False, f"FAIL: open item not in allowed library for node {n['node_id']}"

    return True, "PASS"

def decide_recommendation(nodes: List[TreeNode], prune_log: List[Dict[str, Any]], risks: List[Dict[str, Any]]) -> Dict[str, Any]:
    # Recommend best KEEP branch by overall_score; if none KEEP -> HUMAN_REVIEW
    candidates = []
    for n in nodes:
        if n["label"] == "ROOT":
            continue
        sc = (n.get("derived_metrics") or {}).get("scorecard") or {}
        if n["decision_status"] == "KEEP":
            candidates.append((float(sc.get("overall_score", 0.0)), n))

    if not candidates:
        return {
            "recommended_branch": "HUMAN_REVIEW",
            "decision": "HUMAN_REVIEW",
            "why_chosen": "All branches were pruned under explicit rules; escalation required."
        }

    candidates.sort(key=lambda t: t[0], reverse=True)
    best = candidates[0][1]
    return {
        "recommended_branch": best["label"],
        "decision": "GO",
        "why_chosen": f"Highest overall_score among KEEP branches under deterministic scorecard (node_id={best['node_id']})."
    }

print("Gates + escalation logic ready.")

Gates + escalation logic ready.


##7.TRACE BUILDER

###7.1.OVERVIEW

**CELL 7/10 — Reasoning trace construction and normalization (reasoning_trace.json)**

This cell builds the artifact that makes the pipeline auditable: the reasoning trace. In a normal analysis process, the “trace” lives in the analyst’s head or scattered across drafts. Here, the trace is a formal object that captures the decision topology and each node’s state.

The first step is normalization. Normalization ensures that every node has the same required fields and consistent types. This matters because downstream validation and review rely on structural consistency. Without normalization, small inconsistencies (missing fields, wrong types, null handling) can cause failures or, worse, silent misinterpretation.

Then the cell assembles the full trace object: run_id, timestamp, shape=TREE, root node ID, full node list, prune log, and termination metadata. The inclusion of termination metadata is important for governance because it documents how the run ended: how many nodes were used, what decision was reached, and whether caps were enforced. This is part of the system’s “contract” with reviewers.

Finally, the cell validates the trace against a JSON schema and, if validation fails, it does not proceed quietly. It logs a high-severity schema risk. This is a key learning: schema validation is a control, not a convenience. The trace is the audit trail; if it is invalid, we cannot claim to have a controlled reasoning pipeline.

Cell 7 therefore produces the file that a reviewer would open first to understand “what happened.” It is the structured record of the tree and its evaluation state. In governance terms, it is the difference between “we think we evaluated three options” and “here is the exact machine-readable evidence that we did.”

###7.2.CODE AND IMPLEMENTATION

In [11]:
# CELL 7/10 — Trace builder (reasoning_trace.json) + normalization

REASONING_TRACE_PATH = os.path.join(ARTIFACTS_DIR, "reasoning_trace.json")
RISK_LOG_PATH = os.path.join(ARTIFACTS_DIR, "risk_log.json")

def normalize_node(node: TreeNode) -> TreeNode:
    # Ensure schema fields exist and types normalized
    out: TreeNode = {
        "node_id": str(node.get("node_id")),
        "parent_id": node.get("parent_id", None),
        "label": node.get("label"),  # type: ignore
        "facts_used": list(node.get("facts_used") or []),
        "assumptions": list(node.get("assumptions") or []),
        "derived_metrics": dict(node.get("derived_metrics") or {}),
        "risks": list(node.get("risks") or []),
        "decision_status": node.get("decision_status", "KEEP"),  # type: ignore
        "prune_rationale": node.get("prune_rationale", None)
    }
    if out["decision_status"] != "PRUNE":
        out["prune_rationale"] = None
    return out

def build_trace(run_id: str, nodes: List[TreeNode], prune_log: List[Dict[str, Any]], termination: Dict[str, Any]) -> Dict[str, Any]:
    normalized = [normalize_node(n) for n in nodes]
    trace = {
        "run_id": run_id,
        "timestamp_utc": utc_now_iso(),
        "shape": "TREE",
        "root_node_id": "N0",
        "nodes": normalized,
        "prune_log": prune_log,
        "termination": termination
    }
    return trace

def validate_trace_or_risk(trace: Dict[str, Any], risks: List[Dict[str, Any]]) -> bool:
    ok, err = schema_check(trace, REASONING_TRACE_SCHEMA)
    if not ok:
        risks.append(mk_risk(
            risk_id="GATE_C_SCHEMA_TRACE",
            severity="high",
            category="schema",
            description=f"reasoning_trace.json schema invalid: {err}",
            control="JSON schema validation; force HUMAN_REVIEW",
            status="HUMAN_REVIEW"
        ))
    return ok

print("Trace builder ready.")

Trace builder ready.


##8.REPORT BUILDER

###8.1.OVERVIEW

**CELL 8/10 — Board-facing report composition with strict separation of facts, assumptions, open items**

This cell turns the trace and the case into a board-facing deliverable. The key concept is separation: facts_provided must be included verbatim, assumptions must be explicit, open items must be listed, and the verification status must remain “Not verified.” The report is designed to be readable to senior decision-makers while still remaining control-grade.

The cell constructs a scenario matrix: a compact summary of the three branches with their scorecards and KEEP/PRUNE status. This is the “board table” that supports comparative discussion. It also collects assumptions across branches and deduplicates them, making it obvious what the analysis relied on beyond provided facts. Similarly, it collects open items, which represent what would change the decision if verified.

The report includes an executive summary and a recommendation object. Importantly, the recommendation is not framed as certainty; it includes a confidence level and rationale. In this notebook, confidence is a conservative classification based on score separation and governance posture. This keeps the report honest: it is decision support, not a claim of truth.

Crucially, the cell validates the final report against a JSON schema. If the report fails schema validation, the system logs a high-severity risk and forces a safe fallback recommendation of HUMAN_REVIEW. This is exactly the right behavior: if the system cannot produce a structurally valid report, it cannot ask the board to trust it.

Cell 8 therefore produces the single file that most stakeholders will read: `final_report.json`. But behind that readability is a strict governance structure: it is machine-checkable, explicitly separated into facts/assumptions/open items, and permanently labeled “Not verified” to prevent accidental over-reliance.

###8.2.CODE AND IMPLEMENTATION

In [12]:
# CELL 8/10 — Report composer (final_report.json) with strict schema

FINAL_REPORT_PATH = os.path.join(ARTIFACTS_DIR, "final_report.json")
RUN_MANIFEST_PATH = os.path.join(ARTIFACTS_DIR, "run_manifest.json")

def scenario_matrix(nodes: List[TreeNode]) -> List[Dict[str, Any]]:
    rows = []
    for n in nodes:
        if n["label"] == "ROOT":
            continue
        sc = (n.get("derived_metrics") or {}).get("scorecard") or {}
        rows.append({
            "branch": n["label"],
            "node_id": n["node_id"],
            "decision_status": n["decision_status"],
            "value_score": int(sc.get("value_score", 0)),
            "risk_score": int(sc.get("risk_score", 0)),
            "feasibility_score": int(sc.get("feasibility_score", 0)),
            "overall_score": float(sc.get("overall_score", 0.0))
        })
    return rows

def collect_assumptions(nodes: List[TreeNode]) -> List[str]:
    out = []
    for n in nodes:
        if n["label"] == "ROOT":
            continue
        out.extend(list(n.get("assumptions") or []))
    # stable unique
    seen = set()
    uniq = []
    for a in out:
        if a not in seen:
            uniq.append(a)
            seen.add(a)
    return uniq

def collect_open_items(nodes: List[TreeNode]) -> List[str]:
    out = []
    for n in nodes:
        if n["label"] == "ROOT":
            continue
        oi = (n.get("derived_metrics") or {}).get("open_items") or []
        out.extend(list(oi))
    seen = set()
    uniq = []
    for x in out:
        if x not in seen:
            uniq.append(x)
            seen.add(x)
    return uniq

def compose_report(case: SyntheticCase, nodes: List[TreeNode], prune_log: List[Dict[str, Any]], decision_obj: Dict[str, Any], risks: List[Dict[str, Any]]) -> Dict[str, Any]:
    sm = scenario_matrix(nodes)
    open_items = collect_open_items(nodes)
    assumptions = collect_assumptions(nodes)

    kept = [r for r in sm if r["decision_status"] == "KEEP"]
    confidence_level = "low"
    conf_rationale = "Conservative default under synthetic assumptions; requires verification."
    if decision_obj.get("decision") == "GO" and len(kept) >= 1:
        best = sorted(kept, key=lambda x: x["overall_score"], reverse=True)[0]
        if best["overall_score"] >= 70:
            confidence_level = "medium"
            conf_rationale = "Best branch scores materially ahead under deterministic rubric; still requires verification."
        else:
            confidence_level = "low"
            conf_rationale = "Differences between branches are modest under deterministic rubric; verification required."

    exec_summary = (
        f"Decision tree evaluated three paths (Acquire, Divest, Do nothing) using deterministic scorecards "
        f"and constrained qualitative memos. Recommended path: {decision_obj.get('recommended_branch')} "
        f"subject to open verification items and governance gates."
    )

    report = {
        "run_id": RUN_ID,
        "timestamp_utc": utc_now_iso(),
        "executive_summary": exec_summary,
        "facts_provided": case["facts_provided"],
        "assumptions_introduced": assumptions,
        "analysis": {
            "scoring_method": {
                "weights": CONFIG["scoring_weights"],
                "prune_rules": CONFIG["prune_rules"],
                "note": "Scores are synthetic and deterministic; not market-derived and not investment advice."
            },
            "scenario_matrix": sm
        },
        "recommendation": {
            "recommended_branch": decision_obj.get("recommended_branch"),
            "decision": decision_obj.get("decision"),
            "why_chosen": decision_obj.get("why_chosen")
        },
        "confidence": {
            "level": confidence_level,
            "rationale": conf_rationale
        },
        "open_items": open_items,
        "questions_to_verify": [
            "Which branch assumptions are most sensitive to diligence findings (commercial, operational, financial)?",
            "What is the integration / carve-out readiness evidence and critical path timeline?",
            "Which risks are gating items that must be resolved before proceeding?"
        ],
        "verification_status": "Not verified",
        "scenario_matrix": sm,
        "pruned_branches_audit": prune_log
    }

    ok, err = schema_check(report, FINAL_REPORT_SCHEMA)
    if not ok:
        risks.append(mk_risk(
            risk_id="GATE_C_SCHEMA_REPORT",
            severity="high",
            category="schema",
            description=f"final_report.json schema invalid: {err}",
            control="JSON schema validation; force HUMAN_REVIEW",
            status="HUMAN_REVIEW"
        ))
        # If report invalid, force a minimal safe report that passes schema
        report["recommendation"]["decision"] = "HUMAN_REVIEW"
        report["recommendation"]["recommended_branch"] = "HUMAN_REVIEW"
        report["recommendation"]["why_chosen"] = "Schema validation failed; escalation required."

        ok2, err2 = schema_check(report, FINAL_REPORT_SCHEMA)
        if not ok2:
            raise RuntimeError(f"Report schema still invalid after fallback: {err2}")

    return report

print("Report composer ready.")

Report composer ready.


##9.RUNNING PIPELINE

###9.1.0VERVIEW

**CELL 9/10 — End-to-end orchestration: execute the pipeline, enforce gates, write artifacts**

This cell is the operational run controller. It takes all the components defined earlier—case generation, LLM wrapper, tree engine, gates, trace builder, and report composer—and executes them in the correct order. In a production system, this orchestration function would resemble a batch job or service pipeline; here, it is explicit and teachable.

It begins by creating a run manifest. The manifest records run_id, timestamp, model, config hash, environment fingerprint, determinism settings, and output paths. This is the “who/what/when/how” of the run. It is essential for reproducibility and internal controls. If someone later asks “what version of policy produced this recommendation,” the manifest answers that.

Next, the orchestrator builds the tree and immediately checks branch coverage. Then it calls the LLM once per branch, validates the LLM output against schema, applies additional safety constraints (label enforcement, no digits, library membership filtering), and evaluates nodes. The model’s output is never trusted blindly; it is treated as untrusted until validated and normalized.

After evaluation, the orchestrator prunes branches and runs pruning justification gates and no-invented-facts gates. It then decides on a recommendation: the best KEEP branch by overall score, unless all branches were pruned or any high-severity HUMAN_REVIEW risks exist. That last clause is crucial: governance overrides “best score.” If controls fail, escalation wins.

Finally, the orchestrator writes the reasoning trace, risk log, and final report to disk. It writes the risk log even when problems occur, ensuring failures are recorded. This creates a complete evidence trail for review. Cell 9 is therefore where the notebook becomes a true pipeline: it executes, enforces, logs, and outputs—end-to-end.

###9.2.CODE AND IMPLEMENTATION

In [13]:
# CELL 9/10 — Run orchestrator: executes pipeline end-to-end, writes artifacts

risks: List[Dict[str, Any]] = []

# Run manifest (start)
run_manifest = {
    "run_id": RUN_ID,
    "timestamp_utc": RUN_TS_UTC,
    "project": CONFIG["project"],
    "notebook": CONFIG["notebook"],
    "model": CONFIG["model"],
    "config": CONFIG,
    "config_sha256": sha256_json(CONFIG),
    "environment": {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "base_dir": BASE_DIR
    },
    "determinism": {
        "random_seed": 7,
        "PYTHONHASHSEED": os.environ.get("PYTHONHASHSEED")
    },
    "case_id": CASE["case_id"],
    "verification_status": "Not verified"
}

# Build tree
tree = build_tree(CASE)
nodes = tree["nodes"]

# Gate A (pre): branch coverage must exist
okA, msgA = gate_branch_coverage(nodes)
if not okA:
    risks.append(mk_risk(
        risk_id="GATE_A_BRANCH_COVERAGE",
        severity="high",
        category="governance",
        description=msgA,
        control="Branch coverage check; require exactly 3 branches",
        status="HUMAN_REVIEW"
    ))

# LLM per branch with strict validation
for n in nodes:
    if n["label"] == "ROOT":
        continue
    label = str(n["label"])
    payload = llm_branch_memo(CASE, label)

    ok_payload, err_payload = schema_check(payload, LLM_BRANCH_OUTPUT_SCHEMA)
    if not ok_payload:
        risks.append(mk_risk(
            risk_id=f"LLM_SCHEMA_{n['node_id']}",
            severity="high",
            category="llm_output",
            description=f"LLM branch memo schema invalid for {label}: {err_payload}",
            control="LLM strict JSON schema validation; force HUMAN_REVIEW",
            status="HUMAN_REVIEW"
        ))
        payload = {"label": label, "thesis": "", "assumptions_selected": [], "risks": [], "open_items_selected": []}

    # Additional safety constraints
    if payload.get("label") != label:
        risks.append(mk_risk(
            risk_id=f"LLM_LABEL_MISMATCH_{n['node_id']}",
            severity="medium",
            category="llm_output",
            description=f"LLM label mismatch: expected {label}, got {payload.get('label')}",
            control="Strict label enforcement; override to expected label",
            status="MITIGATED"
        ))
        payload["label"] = label

    # No digits control
    if CONFIG["gates"]["no_digits_in_llm_text"]:
        if _contains_digits(str(payload.get("thesis", ""))) or any(_contains_digits(str(x)) for x in payload.get("risks", [])):
            risks.append(mk_risk(
                risk_id=f"LLM_DIGITS_{n['node_id']}",
                severity="high",
                category="policy",
                description=f"Digits detected in LLM text for {label}; treated as potential invented facts.",
                control="No-digits policy; force HUMAN_REVIEW",
                status="HUMAN_REVIEW"
            ))
            payload["thesis"] = ""
            payload["risks"] = []

    # Library membership enforcement
    allowedA = set(CASE["assumption_library"][label])
    allowedO = set(CASE["open_item_library"][label])

    payload["assumptions_selected"] = [a for a in payload.get("assumptions_selected", []) if a in allowedA]
    payload["open_items_selected"] = [o for o in payload.get("open_items_selected", []) if o in allowedO]

    # Evaluate node
    evaluate_node(n, CASE, payload)

# Prune
nodes, prune_log = prune_tree(nodes)

# Gate B: pruning justification
okB, msgB = gate_pruning_justification(nodes)
if not okB:
    risks.append(mk_risk(
        risk_id="GATE_B_PRUNE_JUSTIFICATION",
        severity="high",
        category="governance",
        description=msgB,
        control="Every PRUNE must cite explicit rule + values",
        status="HUMAN_REVIEW"
    ))

# Gate C: no invented facts detector (library + digits)
okC, msgC = gate_no_invented_facts(nodes, CASE)
if not okC:
    risks.append(mk_risk(
        risk_id="GATE_C_NO_INVENTED_FACTS",
        severity="high",
        category="policy",
        description=msgC,
        control="No invented facts; strict library membership + digits ban",
        status="HUMAN_REVIEW"
    ))

# Decide recommendation
decision_obj = decide_recommendation(nodes, prune_log, risks)

# If any high severity HUMAN_REVIEW risks, force recommendation to HUMAN_REVIEW
if any(r.get("severity") == "high" and r.get("status") == "HUMAN_REVIEW" for r in risks):
    decision_obj["recommended_branch"] = "HUMAN_REVIEW"
    decision_obj["decision"] = "HUMAN_REVIEW"
    decision_obj["why_chosen"] = "One or more governance gates failed; escalation required."

termination = {
    "timestamp_utc": utc_now_iso(),
    "max_nodes_enforced": True,
    "nodes_count": len(nodes),
    "branches_required": CONFIG["gates"]["branch_coverage_required"],
    "decision": decision_obj.get("decision"),
    "recommended_branch": decision_obj.get("recommended_branch")
}

# Build + validate trace
trace = build_trace(RUN_ID, nodes, prune_log, termination)
_ = validate_trace_or_risk(trace, risks)

# Write risk log + trace now (even if report fails)
write_json(REASONING_TRACE_PATH, trace)
write_json(RISK_LOG_PATH, {"run_id": RUN_ID, "timestamp_utc": utc_now_iso(), "risks": risks, "verification_status": "Not verified"})

# Compose report
report = compose_report(CASE, nodes, prune_log, decision_obj, risks)
write_json(FINAL_REPORT_PATH, report)

# Update risk log after report composition
write_json(RISK_LOG_PATH, {"run_id": RUN_ID, "timestamp_utc": utc_now_iso(), "risks": risks, "verification_status": "Not verified"})

# Finalize run manifest
run_manifest["termination"] = termination
run_manifest["outputs"] = {
    "run_manifest": RUN_MANIFEST_PATH,
    "prompts_log": os.path.join(ARTIFACTS_DIR, "prompts_log.jsonl"),
    "reasoning_trace": REASONING_TRACE_PATH,
    "risk_log": RISK_LOG_PATH,
    "final_report": FINAL_REPORT_PATH
}
write_json(RUN_MANIFEST_PATH, run_manifest)

print("Artifacts written:")
for k, v in run_manifest["outputs"].items():
    print(" -", k, "=>", v)
print("Decision:", report["recommendation"])
print("Verification:", report["verification_status"])

Artifacts written:
 - run_manifest => /content/artifacts/run_manifest.json
 - prompts_log => /content/artifacts/prompts_log.jsonl
 - reasoning_trace => /content/artifacts/reasoning_trace.json
 - risk_log => /content/artifacts/risk_log.json
 - final_report => /content/artifacts/final_report.json
Decision: {'recommended_branch': 'HUMAN_REVIEW', 'decision': 'HUMAN_REVIEW', 'why_chosen': 'One or more governance gates failed; escalation required.'}
Verification: Not verified


##10.AUDIT BUNDLE

###10.1.OVERVIEW

**CELL 10/10 — Packaging and delivery: zip the governance bundle and produce a minimal run summary**

This cell performs the final institutional step: it packages the outputs into a deliverable that can be handed to a reviewer, archived, or attached to a governance workflow. In real organizations, the difference between “we generated artifacts somewhere” and “we produced an auditable deliverable package” is significant. Packaging is a control.

The cell defines the list of files that must be included: run manifest, prompts log, reasoning trace, risk log, and final report. These files together tell the complete story: policy, interactions, reasoning structure, governance issues, and final outcome. The cell creates a zip file in `deliverables/deliverables.zip` and writes each artifact with a relative path so the bundle is portable. This is important for sharing: recipients can unzip and preserve the original structure.

After packaging, the cell prints a minimal console summary containing run_id, timestamp, the zip path, and the artifact list. This summary is intentionally small and operational. It is not a narrative. The narrative belongs in the report, while the console output is meant to help the operator confirm that the run completed and where outputs were saved.

The cell also includes notes that reinforce governance posture: synthetic-only, not investment advice, verification_status “Not verified,” and human review required for real decisions. These reminders matter because the final step—sharing outputs—is when accidental over-reliance is most likely.

In short, Cell 10 closes the loop. It ensures the notebook produces not just analysis, but a packaged evidence bundle suitable for governance, committee review, and repeatable recordkeeping. This is how we move from “a notebook that ran” to “a process artifact that can be reviewed and archived.”

###10.2.CODE AND IMPLEMENTATION

####10.2.1.BUNDLE

In [14]:
# CELL 10/10 — Packaging: zip deliverables + minimal console summary of outputs/paths

ZIP_PATH = os.path.join(DELIVERABLES_DIR, "deliverables.zip")

to_zip = [
    os.path.join(ARTIFACTS_DIR, "run_manifest.json"),
    os.path.join(ARTIFACTS_DIR, "prompts_log.jsonl"),
    os.path.join(ARTIFACTS_DIR, "reasoning_trace.json"),
    os.path.join(ARTIFACTS_DIR, "risk_log.json"),
    os.path.join(ARTIFACTS_DIR, "final_report.json"),
]

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in to_zip:
        if os.path.exists(p):
            arc = os.path.relpath(p, BASE_DIR)
            z.write(p, arcname=arc)

summary = {
    "run_id": RUN_ID,
    "timestamp_utc": utc_now_iso(),
    "deliverables_zip": ZIP_PATH,
    "artifact_paths": to_zip,
    "notes": [
        "All outputs are synthetic and governance-first.",
        "verification_status is Not verified; human review required for real decisions."
    ]
}

print(json.dumps(summary, indent=2, ensure_ascii=False, sort_keys=True))

{
  "artifact_paths": [
    "/content/artifacts/run_manifest.json",
    "/content/artifacts/prompts_log.jsonl",
    "/content/artifacts/reasoning_trace.json",
    "/content/artifacts/risk_log.json",
    "/content/artifacts/final_report.json"
  ],
  "deliverables_zip": "/content/deliverables/deliverables.zip",
  "notes": [
    "All outputs are synthetic and governance-first.",
    "verification_status is Not verified; human review required for real decisions."
  ],
  "run_id": "bd625705-c304-4654-a746-c493eae18dc6",
  "timestamp_utc": "2026-02-20T12:07:21.862493+00:00"
}


####10.2.2.THE REPORT IN JSON FORMAT

In [15]:
import json, os

path = "artifacts/final_report.json"
if not os.path.exists(path):
    raise FileNotFoundError(f"Missing {path}. Run the notebook cells 1–10 first.")

with open(path, "r", encoding="utf-8") as f:
    report = json.load(f)

print(json.dumps(report, ensure_ascii=False, indent=2, sort_keys=True))

{
  "analysis": {
    "scenario_matrix": [
      {
        "branch": "Acquire",
        "decision_status": "PRUNE",
        "feasibility_score": 49,
        "node_id": "N1",
        "overall_score": 44.0,
        "risk_score": 82,
        "value_score": 62
      },
      {
        "branch": "Divest",
        "decision_status": "KEEP",
        "feasibility_score": 61,
        "node_id": "N2",
        "overall_score": 59.75,
        "risk_score": 58,
        "value_score": 73
      },
      {
        "branch": "Do nothing",
        "decision_status": "KEEP",
        "feasibility_score": 79,
        "node_id": "N3",
        "overall_score": 56.25,
        "risk_score": 59,
        "value_score": 58
      }
    ],
    "scoring_method": {
      "note": "Scores are synthetic and deterministic; not market-derived and not investment advice.",
      "prune_rules": {
        "max_risk_score": 70,
        "min_feasibility_score": 55
      },
      "weights": {
        "feasibility": 0.2,
        

####10.2.3.THE FINAL REPORT

In [16]:
import json, os, textwrap

path = "artifacts/final_report.json"
if not os.path.exists(path):
    raise FileNotFoundError(f"Missing {path}. Run the notebook cells 1–10 first.")

with open(path, "r", encoding="utf-8") as f:
    r = json.load(f)

case_path = None
# facts_provided are already embedded; the strategic question is in the synthetic case but we can restate it safely.
question = "Strategic question: Buy competitor vs divest division vs do nothing — which path should we recommend as the next step, and why, under governance-first constraints (no external facts; Not verified)?"

rec = r.get("recommendation", {})
decision = rec.get("decision", "HUMAN_REVIEW")
recommended_branch = rec.get("recommended_branch", "HUMAN_REVIEW")
why = rec.get("why_chosen", "")

confidence = r.get("confidence", {})
conf_level = confidence.get("level", "low")
conf_rationale = confidence.get("rationale", "")

facts = r.get("facts_provided", {})
assumptions = r.get("assumptions_introduced", [])
scenario_matrix = r.get("scenario_matrix", [])
pruned = r.get("pruned_branches_audit", [])
open_items = r.get("open_items", [])
questions = r.get("questions_to_verify", [])
verification = r.get("verification_status", "Not verified")

def wrap(s: str, width: int = 100) -> str:
    return "\n".join(textwrap.wrap(s.strip(), width=width)) if s else ""

def fmt_bullets(items, width=100, indent="  - "):
    out = []
    for it in items:
        it_s = str(it)
        lines = textwrap.wrap(it_s, width=width-len(indent))
        if not lines:
            continue
        out.append(indent + lines[0])
        for ln in lines[1:]:
            out.append("    " + ln)
    return "\n".join(out) if out else "  - (none)"

def fmt_matrix(rows):
    # simple fixed columns
    cols = ["branch","decision_status","value_score","risk_score","feasibility_score","overall_score","node_id"]
    header = " | ".join([c.upper().ljust(16) for c in cols])
    sep = "-+-".join(["-"*16 for _ in cols])
    lines = [header, sep]
    for row in rows:
        line = " | ".join([
            str(row.get("branch","")).ljust(16),
            str(row.get("decision_status","")).ljust(16),
            str(row.get("value_score","")).ljust(16),
            str(row.get("risk_score","")).ljust(16),
            str(row.get("feasibility_score","")).ljust(16),
            str(row.get("overall_score","")).ljust(16),
            str(row.get("node_id","")).ljust(16),
        ])
        lines.append(line)
    return "\n".join(lines)

written = f"""
WRITTEN REPORT (GOVERNANCE-FIRST / CONTROL-GRADE)
Run ID: {r.get("run_id")}
Timestamp (UTC): {r.get("timestamp_utc")}
Verification status: {verification}

QUESTION
{wrap(question)}

ANSWER (BOARD-FACING)
Decision: {decision}
Recommended path: {recommended_branch}
Rationale: {wrap(why)}

Confidence: {conf_level}
Confidence rationale: {wrap(conf_rationale)}

EXECUTIVE SUMMARY
{wrap(r.get("executive_summary",""))}

FACTS PROVIDED (SYNTHETIC PACKET — VERBATIM OBJECT)
{json.dumps(facts, ensure_ascii=False, indent=2, sort_keys=True)}

ASSUMPTIONS INTRODUCED (EXPLICIT)
{fmt_bullets(assumptions)}

SCENARIO MATRIX (DETERMINISTIC SCORECARD)
{fmt_matrix(scenario_matrix)}

PRUNED BRANCHES (AUDIT EVIDENCE)
{json.dumps(pruned, ensure_ascii=False, indent=2, sort_keys=True)}

OPEN ITEMS (WHAT WOULD CHANGE THE DECISION)
{fmt_bullets(open_items)}

QUESTIONS TO VERIFY (REQUIRED FOR HUMAN REVIEW)
{fmt_bullets(questions)}

NOTES / CONTROLS
  - No external market data or “facts not provided” were used.
  - Any gate failure forces HUMAN_REVIEW (see artifacts/risk_log.json).
  - This output is synthetic and for teaching/governance demonstration only.
""".strip()

print(written)

WRITTEN REPORT (GOVERNANCE-FIRST / CONTROL-GRADE)
Run ID: bd625705-c304-4654-a746-c493eae18dc6
Timestamp (UTC): 2026-02-20T12:07:04.512139+00:00
Verification status: Not verified

QUESTION
Strategic question: Buy competitor vs divest division vs do nothing — which path should we recommend
as the next step, and why, under governance-first constraints (no external facts; Not verified)?

ANSWER (BOARD-FACING)
Decision: HUMAN_REVIEW
Recommended path: HUMAN_REVIEW
Rationale: One or more governance gates failed; escalation required.

Confidence: low
Confidence rationale: Conservative default under synthetic assumptions; requires verification.

EXECUTIVE SUMMARY
Decision tree evaluated three paths (Acquire, Divest, Do nothing) using deterministic scorecards and
constrained qualitative memos. Recommended path: HUMAN_REVIEW subject to open verification items and
governance gates.

FACTS PROVIDED (SYNTHETIC PACKET — VERBATIM OBJECT)
{
  "baseline_financials": {
    "ebitda_m": 63,
    "ebitda_ma

####10.2.4.NARRATIVE

In [17]:
# Generate a narrative (board-ready) written report using the LLM, grounded ONLY in the saved JSON artifacts.
# This will NOT invent facts: it must quote/use only what is inside final_report.json + reasoning_trace.json + risk_log.json.

!pip -q install "anthropic==0.45.2"

import os, json, re, hashlib, datetime as _dt
from google.colab import userdata
from anthropic import Anthropic

ARTIFACTS_DIR = "artifacts"
FINAL_REPORT_PATH = os.path.join(ARTIFACTS_DIR, "final_report.json")
TRACE_PATH = os.path.join(ARTIFACTS_DIR, "reasoning_trace.json")
RISK_PATH = os.path.join(ARTIFACTS_DIR, "risk_log.json")
PROMPTS_LOG_PATH = os.path.join(ARTIFACTS_DIR, "prompts_log.jsonl")
OUT_TXT = os.path.join("deliverables", "written_report_llm.txt")

def utc_now_iso() -> str:
    return _dt.datetime.now(_dt.timezone.utc).isoformat()

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

_REDACT_PATTERNS = [
    (re.compile(r"sk-[A-Za-z0-9_\-]+"), "[REDACTED_API_KEY]"),
    (re.compile(r"(?i)anthropic_api_key\s*[:=]\s*['\"][^'\"]+['\"]"), "ANTHROPIC_API_KEY='[REDACTED]'"),
    (re.compile(r"(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b"), "[REDACTED_EMAIL]"),
]
def redact_text(s: str) -> str:
    out = s
    for pat, repl in _REDACT_PATTERNS:
        out = pat.sub(repl, out)
    return out

def append_jsonl(path: str, obj: dict) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False, sort_keys=True) + "\n")

for p in [FINAL_REPORT_PATH, TRACE_PATH, RISK_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing {p}. Run notebook cells 1–10 first.")

with open(FINAL_REPORT_PATH, "r", encoding="utf-8") as f:
    final_report = json.load(f)
with open(TRACE_PATH, "r", encoding="utf-8") as f:
    trace = json.load(f)
with open(RISK_PATH, "r", encoding="utf-8") as f:
    risk_log = json.load(f)

api_key = userdata.get("ANTHROPIC_API_KEY")
if not api_key or not isinstance(api_key, str) or len(api_key) < 10:
    raise RuntimeError("Missing/invalid ANTHROPIC_API_KEY in Colab Secrets.")

client = Anthropic(api_key=api_key)

question = (
    "Strategic question: Buy competitor vs divest division vs do nothing. "
    "Provide a board-ready written report that states the question and the answer, "
    "explains the reasoning tree and pruning, and clearly separates facts vs assumptions vs open items. "
    "Use only the provided JSON artifacts; do not introduce any new facts or numbers."
)

payload = {
    "question": question,
    "final_report_json": final_report,
    "reasoning_trace_json": trace,
    "risk_log_json": risk_log,
    "strict_rules": [
        "Do NOT invent any facts, numbers, market data, or sources.",
        "Only use/quote what is present in the provided JSON artifacts.",
        "If information is missing, state it as open items / questions to verify.",
        "Preserve governance framing: verification_status must remain 'Not verified'.",
        "Explain what was pruned and why, referencing prune rules and recorded values from the trace.",
        "Write in clear, board-ready narrative with section headings using plain text (no markdown symbols required)."
    ],
    "required_sections": [
        "TITLE LINE (include run_id and UTC timestamp)",
        "QUESTION",
        "ANSWER (decision + recommended branch + why)",
        "WHAT WE KNOW (facts provided)",
        "WHAT WE ASSUMED (assumptions introduced)",
        "TREE SUMMARY (branches + scorecard + keep/prune + rationale)",
        "PRUNED BRANCHES AUDIT (explicit)",
        "RISKS AND CONTROLS (summarize from risk_log.json)",
        "OPEN ITEMS / QUESTIONS TO VERIFY",
        "VERIFICATION STATUS (Not verified) + HUMAN REVIEW NEXT STEPS"
    ]
}

system = (
    "You are a senior investment banking analyst writing a control-grade memo for a risk committee. "
    "You must be strictly grounded: use ONLY the provided JSON artifacts. "
    "If a detail is not in the artifacts, you must not add it."
)

prompt_text = json.dumps(payload, ensure_ascii=False, sort_keys=True)
prompt_redacted = redact_text(prompt_text)

append_jsonl(PROMPTS_LOG_PATH, {
    "run_id": final_report.get("run_id", "unknown"),
    "timestamp_utc": utc_now_iso(),
    "model": "claude-haiku-4-5-20251001",
    "purpose": "narrative_written_report",
    "prompt_sha256": sha256_text(prompt_redacted),
    "prompt_redacted": prompt_redacted[:2000]
})

msg = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1200,
    temperature=0.0,
    system=system,
    messages=[{"role": "user", "content": prompt_redacted}],
)

narrative = msg.content[0].text if msg and msg.content else ""
narrative_redacted = redact_text(narrative)

append_jsonl(PROMPTS_LOG_PATH, {
    "run_id": final_report.get("run_id", "unknown"),
    "timestamp_utc": utc_now_iso(),
    "model": "claude-haiku-4-5-20251001",
    "purpose": "narrative_written_report_response",
    "response_sha256": sha256_text(narrative_redacted),
    "response_redacted": narrative_redacted[:2000]
})

os.makedirs(os.path.dirname(OUT_TXT), exist_ok=True)
with open(OUT_TXT, "w", encoding="utf-8") as f:
    f.write(narrative_redacted.strip() + "\n")

print(narrative_redacted.strip())
print("\nSaved to:", OUT_TXT)

CONTROL-GRADE MEMORANDUM TO RISK COMMITTEE

TITLE LINE
Strategic Decision Analysis: Acquire vs. Divest vs. Do Nothing
Run ID: bd625705-c304-4654-a746-c493eae18dc6
Timestamp (UTC): 2026-02-20T12:07:04.512139+00:00

---

QUESTION

Should the organization acquire a competitor, divest a division, or take no action?

---

ANSWER

RECOMMENDATION: HUMAN_REVIEW

No single branch has been cleared for execution. The Acquire path has been pruned due to governance gate failures. The Divest and Do Nothing paths remain under consideration but require escalation to senior leadership and board governance for final decision authority.

Recommended Next Step: Escalate to Risk Committee and Board with open verification items resolved before proceeding on any branch.

---

WHAT WE KNOW (FACTS PROVIDED)

Business Profile
- Sector: Industrial services
- End markets: Manufacturing, logistics, utilities
- Cyclicality: Medium
- Revenue: $420 million
- EBITDA: $63 million
- EBITDA margin: 15%
- Net debt: $160 m

##11.CONCLUSION

**Conclusions — Tree Reasoning Under Governance (Chapter 2 Notebook)**

This notebook is a strong starting point because it demonstrates something that most “AI in finance” initiatives fail to do: it turns a strategic analysis exercise into a **controlled, inspectable reasoning pipeline**. In board and committee settings, the core problem is rarely that we cannot generate a memo quickly. The core problem is that we cannot reliably explain—under scrutiny—**how we arrived at a recommendation**, what we assumed, what we did not know, and what should happen next. Chapter 2 directly addresses that problem by implementing Tree Reasoning as an engineered structure: three options, one decision surface, explicit pruning rules, and auditable artifacts.

The most positive contribution is that the notebook makes the **decision topology explicit**. The system does not “think in the dark.” It builds a tree with exactly three branches—Acquire, Divest, Do nothing—and forces each branch to carry a standardized payload: facts used, assumptions, derived metrics, risks, a KEEP/PRUNE status, and (if pruned) a rationale tied to explicit thresholds. This converts what is typically an informal comparison into a **mechanism** that can be inspected and challenged. For a board, that is the difference between “a persuasive narrative” and “a defensible process.”

A second major contribution is the notebook’s **governance posture**. The pipeline is designed to fail safely. It enforces gates that trigger escalation when the process deviates from policy: branch coverage must be complete, every prune must be justified with rule + values, and the system must avoid invented facts by constraining what the LLM can say and select. Importantly, schema enforcement is not treated as a nice-to-have; it is treated as a control. If a schema fails, the system logs the risk and forces HUMAN_REVIEW. That is the correct institutional behavior. In finance, governance is not a slide deck. It is a set of enforceable constraints that shape what the system is allowed to do.

A third contribution is **auditability through artifacts**, produced every run. The notebook creates a run manifest (environment + config hash), a redacted prompts log (hashed + limited), a reasoning trace (tree with node-level data), a risk log (risk_id, severity, control, status), and a final report (board-facing summary and matrix). This bundle is exactly what makes AI outputs reviewable by an independent party. If someone asks, “What did the system see? What did it conclude? What did it assume? Which branch was discarded and why?” we can point to files that answer those questions. In a regulated or litigation-sensitive environment, that is not optional—it is foundational.

A fourth contribution is that the notebook demonstrates the correct role for a large language model in early-stage governance-first systems: **bounded drafting inside hard constraints**. The LLM is not used as a free-form analyst with implicit authority. It is used to generate short qualitative memos while being restricted to selecting assumptions and open items from pre-approved lists. This preserves the LLM’s value—natural language synthesis—while limiting its highest-risk failure mode: plausible fabrication. In other words, the notebook shows a usable hybrid: deterministic scoring for comparability, and constrained narrative for readability.

Those positives matter because they establish a pattern we can scale across future chapters: **make the reasoning shape explicit, bound the input, enforce gates, and produce artifacts.** If the board takes one message from this notebook, it should be this: we are not “buying AI answers.” We are building a **governed reasoning process** that can be supervised.

That said, we should be equally explicit about the limitations and what must improve before any production deployment. The first limitation is that the current notebook relies on **synthetic inputs** and **synthetic scoring**. That is appropriate for a teaching and architecture demonstration, but it is not yet a production decision engine. In real use, the pipeline must ingest bounded internal financials and qualitative evidence under a clear data governance policy, including provenance, approval, and access controls. Without that, the system remains an architecture blueprint rather than an operating tool.

The second limitation is that the “no invented facts” control is intentionally conservative but also blunt. The notebook uses a “no digits in LLM text” policy to reduce numeric invention risk. This is effective as a safety brake, but it is not the final form of claim governance. In production, we need **claim-level grounding controls**, where the model can make quantitative statements only when they are directly tied to fields in facts_provided, and every claim can be traced to evidence. That implies improvements such as: structured citations to internal fields, a “claim registry” that maps sentences to source keys, and automated checks that reject unsupported claims. The present notebook is directionally correct but not yet granular enough.

The third limitation is that the system’s decision policy is still a simplified rubric: value/risk/feasibility scores and explicit prune thresholds. This is appropriate for a controlled demonstration, but in practice we will need more nuanced scoring and policy governance. Boards will reasonably ask: why these weights? why these thresholds? what is the calibration? In future iterations, we should treat the scoring policy as a governed object: versioned, approved, stress-tested, and evaluated for sensitivity. We should also add scenario sensitivity analysis—how often does the recommendation flip if assumptions move within allowed ranges? That will make the recommendation more meaningful, because it will show whether we have a robust decision or a fragile one.

The fourth limitation is that Tree Reasoning is intentionally capped to prevent explosion. That is a feature for governance, but it also means the system currently does not explore second-order contingencies. In real-world work, management may want to see sub-branches (for example, “Acquire with full integration” vs “Acquire with staged integration,” or “Divest via auction” vs “Divest via direct negotiation”). We can add sub-branches later, but only if we maintain strict caps, pruning logic, and trace discipline. The correct evolution is not “add more branches”; it is “add more branches only when the governance controls scale with them.”

The fifth limitation is the current notebook is not yet a complete human-in-the-loop operating model. It escalates to HUMAN_REVIEW when gates fail, but it does not implement the downstream workflow: who reviews, what must be signed off, what changes are permitted, and what gets re-run. In production, escalation must route to a defined review role (analyst lead, risk officer, compliance) with explicit acceptance criteria and a recorded decision (approve/revise/reject). This is not a technical detail—it is the core of institutional accountability.

Because of these limitations, it is critical to stress what this notebook **did** and what it **cannot do yet**.

What it did:
- It implemented a disciplined three-branch decision framework that is explicit, bounded, and reproducible.
- It generated a scenario matrix that compares options under deterministic scoring rules.
- It enforced pruning rules and preserved audit evidence for why branches were pruned.
- It constrained the LLM to drafting inside pre-approved assumption/open-item libraries to reduce fabrication risk.
- It produced a governance bundle (trace, risk log, manifest, report) that supports review and reproducibility.

What it cannot do yet:
- It cannot claim real market valuations, real comps, or any external facts—by design.
- It cannot validate truth; verification remains “Not verified” and requires human diligence.
- It cannot yet do claim-level grounding for every sentence; controls are strong but not fully granular.
- It cannot yet operate as a full end-to-end institutional workflow with approvals, sign-offs, and policy governance over scoring weights and thresholds.
- It cannot yet incorporate real evidence attachments (contracts, pipeline data, customer cohorts) with provenance and access control.

This clarity is not a weakness; it is the governance stance that makes the system safe to discuss with a board. The board should see that the system is intentionally limited today, because uncontrolled capability without controls is precisely what creates risk.

Finally, this notebook provides a natural bridge to future chapters because it establishes the second of five reasoning shapes. Tree Reasoning answers the board’s comparative question: **which strategic path is directionally preferable under explicit assumptions and constraints?** The next chapters extend the same governance-first principles into more complex operational realities:

- The **Loop Reasoning** chapter will show how we iterate: how a memo improves over multiple passes, how convergence is defined, and how we stop safely when uncertainty remains. This bridges from “compare options once” to “refine the decision as new information arrives.”
- The **Committee Reasoning** chapter will show how we preserve dissent and enforce quorum. This bridges from “one system recommendation” to “multi-role governance,” mirroring how actual institutions decide.
- The **Trainable Reasoning** chapter will show how we measure and improve quality safely—treating prompt adaptation as an evaluated change, not mythology. This bridges from “static policy” to “controlled evolution.”

In other words, Chapter 2 is the first place where the system behaves like an institutional decision tool: it builds an explicit structure, produces auditable results, and fails safely. The future chapters do not replace this; they build on it—adding iteration, role-based disagreement, and evaluated improvement while maintaining the same core promise: **traceable reasoning under governance.**

##12.playground